<div style="text-align:center;background:linear-gradient(135deg,#0062ff 0%,#00d4ff 100%);font-family:'Segoe UI',Roboto,Arial,sans-serif;color:white;padding:35px 20px;border-radius:15px;box-shadow:0 10px 25px rgba(0,98,255,0.3);margin-bottom:25px;">
  <div style="font-size:35px;font-weight:800;letter-spacing:1.5px;text-transform:uppercase;line-height:1.2;">
    Trực Quan Hóa Dữ Liệu - Đồ án cuối kỳ
  </div>
  <div style="font-size:16px;font-weight:500;margin-top:10px;font-style:italic;opacity:0.9;">
    "Bức tranh thị trường việc làm IT Việt Nam: Kỹ năng, Mức lương và Xu hướng tuyển dụng"
  </div>
  <div style="font-size:18px;font-weight:600;margin-top:15px;border-top:1px solid rgba(255,255,255,0.4);display:inline-block;padding-top:10px;letter-spacing:1px;">
    NHÓM 05 - FIT-HCMUS
  </div>
</div>

<div style="text-align:center;font-size:36px;font-weight:bold;">THU THẬP DỮ LIỆU VIỆC LÀM IT VIỆT NAM</div>

## **1. Cài đặt thư viện và chuẩn bị**

### **1.1. Mục tiêu**
Cài đặt toàn bộ thư viện cần thiết cho pipeline thu thập dữ liệu gồm 4 nguồn:
VietJobs (HuggingFace), Wayback Machine (ITviec/TopDev/TopCV/JobsGO/Vieclam24h),
HTML thủ công, và bước gộp cuối cùng.

### **1.2. Thư viện sử dụng**
* `aiohttp` + `nest_asyncio`: HTTP bất đồng bộ cho Wayback crawler trong Jupyter.
* `beautifulsoup4`: Parse HTML listing page (ITviec, JobsGO, Vieclam24h).
* `datasets` + `huggingface_hub`: Tải VietJobs từ HuggingFace Hub.
* `pandas` + `matplotlib`: Thống kê và vẽ biểu đồ phân bố bộ dữ liệu.

### **1.3. Quy trình**
Chạy cell dưới đây một lần duy nhất để cài đặt. Sau đó chạy từng section theo thứ tự.

In [ ]:
import sys
!{sys.executable} -m pip install aiohttp beautifulsoup4 datasets huggingface_hub nest_asyncio tqdm pandas matplotlib
print('\nCài đặt hoàn tất.')

## **2. Tải dataset VietJobs từ HuggingFace**

### **2.1. Mục tiêu**
Tải dataset **VietJobs** (dinhieufam/VietJobs, 48,092 tin tuyển dụng toàn quốc, CC BY 4.0)
và lọc lấy các job thuộc ngành IT. Đây là nguồn dữ liệu lớn nhất, có đầy đủ salary, skills, location, nhưng **không có posting_date**.

### **2.2. Cấu hình**
* **Dataset**: `dinhieufam/VietJobs` trên HuggingFace Hub.
* **IT category**: `công_nghệ_thông_tin_kỹ_thuật_số` (~1,906 rows).
* **HuggingFace token**: cần tài khoản miễn phí tại https://huggingface.co,
  vào Settings → Access Tokens → New token (Read).

### **2.3. Quy trình**
1. Load dataset dạng streaming (không cần download toàn bộ ~1 GB).
2. Với mỗi row: kiểm tra `category` → nếu IT thì normalize sang schema chung.
3. Lưu ra `../crawler/output/vietjobs_it.json`.

In [ ]:
"""
Section 2: Tải VietJobs từ HuggingFace Hub (streaming, CC BY 4.0).
"""
import json
from pathlib import Path

OUTPUT_DIR = Path("../crawler/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IT_CATEGORIES = {
    "công_nghệ_thông_tin_kỹ_thuật_số",
    "IT & Digital Engineering",
    "Information Technology",
    "Software",
    "Technology",
}

def normalize_vietjobs_row(row: dict) -> dict:
    skills = []
    for col in ("technical_skills", "soft_skills"):
        val = row.get(col) or ""
        if isinstance(val, list):
            skills.extend([s.strip() for s in val if s.strip()])
        elif isinstance(val, str) and val.strip():
            skills.extend([s.strip() for s in val.split(",") if s.strip()])

    def to_m(v):
        if v is None: return None
        try: v = float(str(v).replace(",", "").strip())
        except: return None
        return round(v / 1_000_000, 2) if v > 1000 else round(v, 2)

    sal_min = to_m(row.get("salary_min"))
    sal_max = to_m(row.get("salary_max"))
    if sal_min and sal_max:
        salary_text = f"{sal_min:.0f}\u2013{sal_max:.0f} tri\u1ec7u"
    elif sal_max:
        salary_text = f"\u0111\u1ebfn {sal_max:.0f} tri\u1ec7u"
    elif sal_min:
        salary_text = f"t\u1eeb {sal_min:.0f} tri\u1ec7u"
    else:
        salary_text = row.get("salary") or None

    return {
        "job_title":           row.get("job_title"),
        "company_name":        None,
        "location":            row.get("location"),
        "salary":              salary_text,
        "salary_min":          sal_min,
        "salary_max":          sal_max,
        "skills":              skills if skills else None,
        "description":         (row.get("description") or "")[:500] or None,
        "benefits":            (row.get("benefits") or "")[:300] or None,
        "employment_type":     row.get("contract_type"),
        "industry":            row.get("category"),
        "experience_required": row.get("experience_required"),
        "posting_date":        None,
        "deadline":            None,
        "source":              "VietJobs",
        "url":                 None,
        "crawled_at":          None,
    }

def download_vietjobs(token: str = "", repo_id: str = "dinhieufam/VietJobs") -> list:
    from datasets import load_dataset
    print(f"  Loading {repo_id} (streaming)...")
    kwargs = dict(split="train", streaming=True)
    if token: kwargs["token"] = token
    ds = load_dataset(repo_id, **kwargs)
    jobs, total = [], 0
    for row in ds:
        total += 1
        cat = row.get("category") or ""
        if any(it.lower() in cat.lower() for it in IT_CATEGORIES):
            jobs.append(normalize_vietjobs_row(row))
            if len(jobs) % 200 == 0:
                print(f"  Scanned {total} rows, {len(jobs)} IT jobs...")
    print(f"  Done: {total} rows scanned, {len(jobs)} IT jobs")
    return jobs

# ── CHẠY ─────────────────────────────────────────────────────────
HF_TOKEN = ""  # ← Điền HuggingFace Read token tại đây
               # Lấy tại: https://huggingface.co/settings/tokens

vj_jobs = download_vietjobs(HF_TOKEN)

out_file = OUTPUT_DIR / "vietjobs_it.json"
with open(out_file, "w", encoding="utf-8") as f:
    json.dump(vj_jobs, f, ensure_ascii=False, indent=2)

print(f"\nSaved {len(vj_jobs)} jobs → {out_file}")
print(f"  Có salary_min : {sum(1 for j in vj_jobs if j.get('salary_min'))}")
print(f"  Có skills     : {sum(1 for j in vj_jobs if j.get('skills'))}")

## **3. Crawler Wayback Machine**

### **3.1. Mục tiêu**
Lấy dữ liệu lịch sử từ 5 trang tuyển dụng IT thông qua **Wayback Machine** (Internet Archive). Mỗi trang được archived theo từng snapshot;
ta truy vấn CDX API để tìm URL, rồi fetch HTML bất đồng bộ và extract dữ liệu.

### **3.2. Cơ chế hoạt động**
* **CDX API** (`web.archive.org/cdx/search/cdx`): Tìm danh sách URL đã archived với `matchType=prefix`, `collapse=urlkey` (1 snapshot/URL), `filter=statuscode:200`.
* **`if_` flag**: Fetch `web.archive.org/web/{timestamp}if_/{url}` → trả về HTML gốc không có Wayback toolbar, nhanh hơn và sạch hơn.
* **JSON-LD** (`schema.org/JobPosting`): ITviec, TopDev, TopCV, Vieclam24h nhúng structured data trong `<script type='application/ld+json'>` → extract trực tiếp.
* **aiohttp + asyncio.Semaphore**: Fetch song song có giới hạn (concurrency=10) để không bị Wayback rate-limit.

### **3.3. Các nguồn dữ liệu**
| Nguồn | Prefix CDX | IT filter |
|-------|-----------|----------|
| ITviec | `itviec.com/it-jobs/` | Toàn bộ IT |
| TopDev | `topdev.vn/detail-jobs/` | Toàn bộ IT |
| TopCV | `topcv.vn/viec-lam/` | Slug filter (keyword IT) |
| Vieclam24h | `vieclam24h.vn/it-` | Path filter `/it-phan-mem/` |

### **3.4. Quy trình thực thi**
1. Chạy **cell 3A** để định nghĩa toàn bộ utility functions và cấu hình SOURCES.
2. Chạy **cell 3B** để định nghĩa hàm CDX, async fetch và `main()`.
3. Chạy **từng cell nguồn** (3C–3G) riêng lẻ — mỗi cell tự save file JSON độc lập.

In [ ]:
"""
Cell 3A: Shared utilities và SOURCES config cho Wayback crawler.
"""
import re, json, asyncio, time, urllib.parse, urllib.request
from datetime import datetime
from pathlib import Path
from bs4 import BeautifulSoup

OUTPUT_DIR = Path("../crawler/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CDX_API      = "https://web.archive.org/cdx/search/cdx"
WAYBACK_BASE = "https://web.archive.org/web"

# ── Cấu hình các nguồn ────────────────────────────────────────
SOURCES = {
    "itviec": {
        "name": "ITviec",
        "cdx_prefix": "itviec.com/it-jobs/",
        "url_re": re.compile(r"itviec\.com/it-jobs/[\w\-]+-\d+"),
        "it_slug_filter": None,
        "exclude_slug_filter": None,
        "out_file": "wayback_itviec.json",
        "source_label": "ITviec (Wayback)",
    },
    "topdev": {
        "name": "TopDev",
        "cdx_prefix": "topdev.vn/detail-jobs/",
        "url_re": re.compile(r"topdev\.vn/detail-jobs/[\w\-]+-\d+"),
        "it_slug_filter": None,
        "exclude_slug_filter": None,
        "out_file": "wayback_topdev.json",
        "source_label": "TopDev (Wayback)",
    },
    "topcv": {
        "name": "TopCV",
        "cdx_prefix": "topcv.vn/viec-lam/",
        "url_re": re.compile(r"topcv\.vn/viec-lam/[\w\-]+/\d+\.html"),
        "it_slug_filter": re.compile(
            r"backend|frontend|developer|engineer|devops|software|"
            r"lap-trinh|mobile|android|ios-|fullstack|full-stack|"
            r"tester|qa-|qc-|cloud-|machine-learning|blockchain|"
            r"nodejs|react-|angular-|vue-js|php-developer|golang|"
            r"flutter|database|sre-|security-engineer|phan-mem|"
            r"data-analyst|data-scientist|data-engineer|it-support|"
            r"ky-su-phan-mem|javascript|python-developer|java-developer",
            re.I,
        ),
        "exclude_slug_filter": None,
        "out_file": "wayback_topcv.json",
        "source_label": "TopCV (Wayback)",
    },
    "vieclam24h": {
        "name": "Vieclam24h",
        "cdx_prefix": "vieclam24h.vn/it-",
        "url_re": re.compile(
            r"vieclam24h\.vn/it-(?:phan-mem|phan-cung-mang)/[\w\-]+-c\d+p\d+id\d+\.html"
        ),
        "it_slug_filter": None,
        "exclude_slug_filter": re.compile(
            r"telesale|kinh-doanh-sale|nhan-vien-sale|"
            r"cham-soc-khach-hang|cskh-|ke-toan|hanh-chinh|thu-ky|"
            r"nhan-su|bao-ve|tap-vu|van-phong",
            re.I,
        ),
        "out_file": "wayback_vieclam24h.json",
        "source_label": "Vieclam24h (Wayback)",
    },
}

# ── Shared parser functions ────────────────────────────────────
def extract_jsonld(html: str) -> dict | None:
    pat = re.compile(r'<script[^>]+application/ld\+json[^>]*>(.*?)</script>', re.DOTALL)
    for raw in pat.findall(html):
        try:
            data = json.loads(raw)
            items = data if isinstance(data, list) else [data]
            for item in items:
                if item.get("@type") == "JobPosting":
                    return item
        except Exception:
            continue
    return None

def clean_html_text(s):
    if not s: return None
    return BeautifulSoup(s, "html.parser").get_text(" ", strip=True)[:500]

def _parse_vnd_number(s: str):
    s = re.sub(r"[^\d.,]", "", s)
    if s.count(".") > 1:    s = s.replace(".", "")
    elif s.count(",") > 1: s = s.replace(",", "")
    elif "." in s and "," in s: s = s.replace(",", "")
    elif "," in s and s.index(",") == len(s)-4: s = s.replace(",", "")
    try: return float(s)
    except ValueError: return None

def parse_salary(base_salary):
    if not base_salary: return None, None, None
    val = base_salary.get("value", {})
    if not isinstance(val, dict): return None, None, None
    currency = base_salary.get("currency", "VND")
    min_v = val.get("minValue"); max_v = val.get("maxValue"); single = val.get("value")
    def to_m(v, curr):
        if v is None: return None
        if isinstance(v, str): v = _parse_vnd_number(v)
        if v is None: return None
        try: v = float(v)
        except: return None
        return round(v * 25/1000, 2) if curr == "USD" else round(v/1_000_000, 2)
    if min_v is not None or max_v is not None:
        mn, mx = to_m(min_v, currency), to_m(max_v, currency)
        if mn and mx:  return f"{mn:.0f}\u2013{mx:.0f} tri\u1ec7u", mn, mx
        elif mx:       return f"\u0111\u1ebfn {mx:.0f} tri\u1ec7u", None, mx
        elif mn:       return f"t\u1eeb {mn:.0f} tri\u1ec7u", mn, None
    if isinstance(single, str):
        text = single.strip()
        neg = {"negotiable","thương lượng","thoả thuận","thoa thuan","competitive"}
        if any(w in text.lower() for w in neg): return text, None, None
        nums_raw = re.findall(r"[\d][.\d,]*[\d]", text)
        nums = [v for n in nums_raw if (v := _parse_vnd_number(n)) and v >= 1]
        if len(nums) >= 2:
            return text, to_m(min(nums), currency), to_m(max(nums), currency)
        elif len(nums) == 1:
            up_to = re.search(r"up to|t\u1ed1i \u0111a|\u0111\u1ebfn", text, re.I)
            m = to_m(nums[0], currency)
            return text, (None if up_to else m), m
        return text, None, None
    return None, None, None

def parse_location(job_location):
    if not job_location: return None
    if isinstance(job_location, list):
        locs = []
        for loc in job_location:
            addr = loc.get("address", {})
            city = addr.get("addressLocality") or addr.get("addressRegion")
            if city: locs.append(city)
        return ", ".join(locs) if locs else None
    addr = job_location.get("address", {})
    if isinstance(addr, list): addr = addr[0] if addr else {}
    if not isinstance(addr, dict): return None
    return addr.get("addressLocality") or addr.get("addressRegion")

def parse_date(s):
    if not s: return None
    m = re.search(r"(\d{4}-\d{2}-\d{2})", str(s))
    return m.group(1) if m else None

def wayback_ts_to_date(ts: str):
    if ts and len(ts) >= 8:
        return f"{ts[:4]}-{ts[4:6]}-{ts[6:8]}"
    return None

def normalize_job(jld: dict, source: str, url: str, snapshot_date) -> dict:
    salary_text, sal_min, sal_max = parse_salary(jld.get("baseSalary"))
    org = jld.get("hiringOrganization", {})
    skills_raw = jld.get("skills", "")
    if isinstance(skills_raw, list):
        skills = [s.strip() for s in skills_raw if s.strip()]
    else:
        skills = [s.strip() for s in str(skills_raw).split(",") if s.strip()]
    emp_type = jld.get("employmentType", "")
    if isinstance(emp_type, list): emp_type = ", ".join(emp_type)
    posting_date = parse_date(jld.get("datePosted")) or snapshot_date
    return {
        "job_title":       jld.get("title"),
        "company_name":    org.get("name") if isinstance(org, dict) else None,
        "location":        parse_location(jld.get("jobLocation")),
        "salary":          salary_text,
        "salary_min":      sal_min,
        "salary_max":      sal_max,
        "skills":          skills if skills else None,
        "description":     clean_html_text(jld.get("description")),
        "benefits":        clean_html_text(jld.get("jobBenefits")),
        "employment_type": emp_type or None,
        "industry":        jld.get("industry"),
        "posting_date":    posting_date,
        "snapshot_date":   snapshot_date,
        "deadline":        parse_date(jld.get("validThrough")),
        "source":          source,
        "url":             url,
        "crawled_at":      datetime.now().isoformat(),
    }

print("Cell 3A: Utilities và SOURCES config đã sẵn sàng.")

In [ ]:
"""
Cell 3B: CDX API query, async Wayback fetch, và hàm main().
Cần chạy Cell 3A trước.
"""
import aiohttp
import nest_asyncio
nest_asyncio.apply()

WB_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,*/*;q=0.8",
    "Accept-Language": "vi-VN,vi;q=0.9,en;q=0.8",
}

_SEM = None

def fetch_cdx_urls(src_cfg: dict, from_ts: str, to_ts: str, limit: int) -> list:
    """
    Truy vấn CDX API → list of (original_url, wayback_timestamp).
    Dedup theo URL gốc; áp dụng IT slug filter và exclude filter.
    """
    params = {
        "url":       src_cfg["cdx_prefix"],
        "output":    "json",
        "fl":        "original,timestamp",
        "from":      from_ts,
        "to":        to_ts,
        "collapse":  "urlkey",
        "limit":     limit * 5,
        "matchType": "prefix",
        "filter":    "statuscode:200",
    }
    req_url = f"{CDX_API}?{urllib.parse.urlencode(params)}"
    print(f"  CDX query: {req_url[:130]}...")
    try:
        with urllib.request.urlopen(req_url, timeout=60) as resp:
            rows = json.loads(resp.read().decode("utf-8"))
    except Exception as e:
        print(f"  CDX request failed: {e}"); return []
    if not rows or len(rows) < 2:
        print("  CDX returned 0 results"); return []

    it_re   = src_cfg["it_slug_filter"]
    excl_re = src_cfg.get("exclude_slug_filter")
    url_re  = src_cfg["url_re"]
    results, seen, skipped = [], set(), 0
    for row in rows[1:]:
        if len(row) < 2: continue
        orig_url, ts = row[0], row[1]
        if not url_re.search(orig_url): continue
        if it_re and not it_re.search(orig_url):   skipped += 1; continue
        if excl_re and excl_re.search(orig_url):   skipped += 1; continue
        if orig_url in seen: continue
        seen.add(orig_url)
        results.append((orig_url, ts))
    if skipped: print(f'  CDX: skipped {skipped} non-IT slugs')
    print(f'  CDX: {len(results)} unique IT job URLs found')
    return results[:limit * 2]

async def _fetch_one(session, orig_url, ts, total, counters, results, lock, src_label):
    wayback_url = f"{WAYBACK_BASE}/{ts}if_/{orig_url}"
    assert _SEM is not None
    async with _SEM:
        try:
            async with session.get(wayback_url, timeout=aiohttp.ClientTimeout(total=20)) as resp:
                if resp.status != 200:
                    async with lock: counters["fail"] += 1
                    return
                html = await resp.text(errors="replace")
        except Exception:
            async with lock: counters["fail"] += 1
            return

        if len(html) < 5000:
            async with lock: counters["fail"] += 1
            return

        jld = extract_jsonld(html)
        if not jld:
            async with lock: counters["no_ld"] += 1
            return

        job = normalize_job(jld, src_label, orig_url, wayback_ts_to_date(ts))
        async with lock:
            results.append(job)
            counters["ok"] += 1
            done = counters["ok"] + counters["fail"] + counters["no_ld"]
            if counters["ok"] % 50 == 0 or done <= 3:
                sal = job.get("salary") or "–"
                sk  = len(job.get("skills") or [])
                dt  = job.get("posting_date") or "?"
                print(f"  [{done}/{total}] ✓ {str(job.get('job_title',''))[:35]} | {dt} | {sal[:15]} | {sk} skills")
        await asyncio.sleep(0.3)

async def fetch_wayback_jobs(entries: list, concurrency: int, src_label: str) -> list:
    global _SEM
    _SEM = asyncio.Semaphore(concurrency)
    results, counters, lock = [], {"ok": 0, "fail": 0, "no_ld": 0}, asyncio.Lock()
    connector = aiohttp.TCPConnector(limit=concurrency, ssl=False)
    print(f'  Fetching {len(entries)} archived pages (concurrency={concurrency})...')
    async with aiohttp.ClientSession(headers=WB_HEADERS, connector=connector) as session:
        tasks = [_fetch_one(session, u, ts, len(entries), counters, results, lock, src_label)
                 for u, ts in entries]
        await asyncio.gather(*tasks)
    print(f"\n  Done: {counters['ok']} ok | {counters['fail']} fail | {counters['no_ld']} no-jsonld")
    return results

async def run_wayback(source: str, from_ym: str = "202301", to_ym: str = "202606",
                      max_jobs: int = 1000, concurrency: int = 10):
    """Chạy crawler cho một nguồn cụ thể, save ra file JSON."""
    src_cfg = SOURCES[source]
    from_ts = from_ym + "01000000"
    to_ts   = to_ym   + "31235959"
    print(f"Wayback crawler — {src_cfg['name']}")
    print(f'  Period: {from_ym} → {to_ym} | Max: {max_jobs} | Concurrency: {concurrency}\n')

    print("Step 1: Query CDX API...")
    entries = fetch_cdx_urls(src_cfg, from_ts, to_ts, max_jobs)
    if not entries:
        print("No URLs found."); return
    entries = entries[:max_jobs]
    print(f'  → {len(entries)} URLs to fetch\n')

    print("Step 2: Fetch archived pages...")
    t0 = time.time()
    jobs = await fetch_wayback_jobs(entries, concurrency, src_cfg["source_label"])
    elapsed = time.time() - t0

    if not jobs:
        print("No jobs extracted."); return

    jobs.sort(key=lambda j: j.get("posting_date") or "", reverse=True)
    out_file = OUTPUT_DIR / src_cfg["out_file"]
    with open(out_file, "w", encoding="utf-8") as f:
        json.dump(jobs, f, ensure_ascii=False, indent=2)

    has_sal  = sum(1 for j in jobs if j.get("salary_min"))
    has_sk   = sum(1 for j in jobs if j.get("skills"))
    months   = {}
    for j in jobs:
        ym = (j.get("posting_date") or "")[:7] or "unknown"
        months[ym] = months.get(ym, 0) + 1
    bar_max = max(months.values()) if months else 1

    print(f"\n{'='*55}")
    print(f"RESULT — {src_cfg['name']}: {len(jobs)} jobs ({elapsed:.0f}s)")
    print(f'  Có salary_min: {has_sal} | Có skills: {has_sk}')
    print(f"\nPhân bố theo tháng:")
    for ym, cnt in sorted(months.items()):
        bar = '█' * min(40, cnt * 40 // bar_max)
        print(f'  {ym}: {cnt:4d} {bar}')
    print(f'\nSaved: {out_file}')

print("Cell 3B: CDX + async fetch functions đã sẵn sàng.")

### **3.5. Chạy crawler cho từng nguồn**

Mỗi cell dưới đây chạy độc lập. Kết quả tự động lưu vào `../crawler/output/`.

**Thời gian ước tính:**
| Nguồn | Jobs | Thời gian |
|-------|------|----------|
| ITviec | ~4,000 | 15–25 phút |
| TopDev | ~700 | 5–10 phút |
| TopCV | ~170 | 2–4 phút |
| Vieclam24h | ~2,000 | 10–20 phút |

> **Lưu ý:** Nếu file đã tồn tại trong `../crawler/output/`, bỏ qua cell tương ứng.

In [ ]:
# Cell 3C: Crawler ITviec — cần chạy Cell 3A + 3B trước
from pathlib import Path
_out = Path("../crawler/output/wayback_itviec.json")
if _out.exists() and _out.stat().st_size > 1000:
    print(f"File đã tồn tại: {_out.name} ({_out.stat().st_size // 1024} KB) — bỏ qua crawl.")
else:
    try:
        asyncio.run(run_wayback(
            source="itviec",
            from_ym="202301",
            to_ym="202606",
            max_jobs=5000,
            concurrency=10,
        ))
    except Exception as e:
        print(f"Crawler gặp lỗi: {e}")
        print("Bỏ qua nguồn này. Dữ liệu từ các nguồn khác vẫn đủ để chạy.")

In [ ]:
# Cell 3D: Crawler TopDev — cần chạy Cell 3A + 3B trước
from pathlib import Path
_out = Path("../crawler/output/wayback_topdev.json")
if _out.exists() and _out.stat().st_size > 1000:
    print(f"File đã tồn tại: {_out.name} ({_out.stat().st_size // 1024} KB) — bỏ qua crawl.")
else:
    try:
        asyncio.run(run_wayback(
            source="topdev",
            from_ym="202301",
            to_ym="202606",
            max_jobs=2000,
            concurrency=10,
        ))
    except Exception as e:
        print(f"Crawler gặp lỗi: {e}")
        print("Bỏ qua nguồn này. Dữ liệu từ các nguồn khác vẫn đủ để chạy.")

In [ ]:
# Cell 3E: Crawler TopCV — cần chạy Cell 3A + 3B trước
from pathlib import Path
_out = Path("../crawler/output/wayback_topcv.json")
if _out.exists() and _out.stat().st_size > 1000:
    print(f"File đã tồn tại: {_out.name} ({_out.stat().st_size // 1024} KB) — bỏ qua crawl.")
else:
    try:
        asyncio.run(run_wayback(
            source="topcv",
            from_ym="202301",
            to_ym="202606",
            max_jobs=3000,
            concurrency=10,
        ))
    except Exception as e:
        print(f"Crawler gặp lỗi: {e}")
        print("Bỏ qua nguồn này. Dữ liệu từ các nguồn khác vẫn đủ để chạy.")

In [ ]:
# Cell 3F: Crawler Vieclam24h — cần chạy Cell 3A + 3B trước
from pathlib import Path
_out = Path("../crawler/output/wayback_vieclam24h.json")
if _out.exists() and _out.stat().st_size > 1000:
    print(f"File đã tồn tại: {_out.name} ({_out.stat().st_size // 1024} KB) — bỏ qua crawl.")
else:
    try:
        asyncio.run(run_wayback(
            source="vieclam24h",
            from_ym="202301",
            to_ym="202606",
            max_jobs=3000,
            concurrency=10,
        ))
    except Exception as e:
        print(f"Crawler gặp lỗi: {e}")
        print("Bỏ qua nguồn này. Dữ liệu từ các nguồn khác vẫn đủ để chạy.")

## **4. Parser HTML thủ công**

### **4.1. Mục tiêu**
Một số trang (ITviec, JobsGO, Vieclam24h) chặn crawl tự động. Giải pháp: **copy thủ công** HTML của trang listing → để parser tự extract.

### **4.2. Cách lấy HTML**
1. Vào trang tuyển dụng (ví dụ `itviec.com/it-jobs/`).
2. Nhấn **Ctrl+U** → mở tab HTML → **Ctrl+A** → **Ctrl+C**.
3. Paste vào file `.html`, đặt tên theo trang: `itviec_page1.html`, `jobsgo_page1.html`...
4. Lưu vào thư mục `../crawler/html_input/`.
5. Chạy cell bên dưới.

### **4.3. Site detection**
Parser tự nhận dạng site qua tên file và nội dung HTML, rồi dùng parser phù hợp:
* **ITviec**: tìm `class='job-card'` → extract title, company, location, salary, skills.
* **JobsGO**: tìm `div.job-card[data-id]` → extract từ spans và badges.
* **Vieclam24h**: tìm `a[data-job-id]` → extract từ h3 tags, blue spans.
* **Fallback**: tìm `schema.org/JobPosting` JSON-LD (dùng cho detail pages).

In [ ]:
"""
Section 4: Parse HTML thủ công từ các trang tuyển dụng.
"""
import re, json
from datetime import datetime, timedelta
from pathlib import Path
from bs4 import BeautifulSoup

INPUT_DIR  = Path("../crawler/html_input")
OUTPUT_DIR = Path("../crawler/output")
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── JSON-LD fallback ──────────────────────────────────────────
def extract_jsonld_jobs(html: str) -> list:
    results = []
    for raw in re.findall(r'<script[^>]+application/ld\+json[^>]*>(.*?)</script>', html, re.DOTALL):
        try:
            data = json.loads(raw)
            items = data if isinstance(data, list) else [data]
            for item in items:
                if item.get("@type") == "JobPosting": results.append(item)
        except Exception: continue
    return results

def normalize_jsonld(jld: dict, source: str) -> dict:
    org = jld.get("hiringOrganization", {})
    skills_raw = jld.get("skills", "")
    skills = [s.strip() for s in (skills_raw if isinstance(skills_raw, list) else str(skills_raw).split(",")) if s.strip()]
    loc = jld.get("jobLocation")
    location = None
    if isinstance(loc, list) and loc: loc = loc[0]
    if isinstance(loc, dict):
        addr = loc.get("address", {})
        if isinstance(addr, list): addr = addr[0] if addr else {}
        if isinstance(addr, dict):
            location = addr.get("addressLocality") or addr.get("addressRegion")
    base = jld.get("baseSalary", {})
    salary_text = salary_min = salary_max = None
    if isinstance(base, dict):
        val = base.get("value", {}); currency = base.get("currency", "VND")
        if isinstance(val, dict):
            def to_m(v):
                if v is None: return None
                try: v = float(v)
                except: return None
                return round(v * 25/1000, 2) if currency == "USD" else round(v/1_000_000, 2)
            mn, mx = to_m(val.get("minValue")), to_m(val.get("maxValue"))
            if mn and mx: salary_text = f"{mn:.0f}\u2013{mx:.0f} tri\u1ec7u"; salary_min, salary_max = mn, mx
            elif isinstance(val.get("value"), str): salary_text = val["value"].strip()
    desc = jld.get("description", "")
    if desc: desc = BeautifulSoup(desc, "html.parser").get_text(" ", strip=True)[:500]
    return {
        "job_title": jld.get("title"), "company_name": org.get("name") if isinstance(org, dict) else None,
        "location": location, "salary": salary_text, "salary_min": salary_min, "salary_max": salary_max,
        "skills": skills or None, "description": desc or None, "employment_type": jld.get("employmentType"),
        "posting_date": (lambda s: (re.search(r"(\d{4}-\d{2}-\d{2})", str(s)) or [None, None])[1] if s else None)(jld.get("datePosted")),
        "source": source, "url": jld.get("url"), "crawled_at": datetime.now().isoformat(),
    }

# ── ITviec listing parser ─────────────────────────────────────
def parse_itviec_listing(html: str, filename: str) -> list:
    soup = BeautifulSoup(html, "html.parser")
    cards = soup.find_all(class_="job-card")
    jobs = []
    for card in cards:
        h3 = card.find("h3"); title = h3.get_text(strip=True) if h3 else None
        slug = card.get("data-search--job-selection-job-slug-value", "")
        url = f"https://itviec.com/it-jobs/{slug}" if slug else None
        company = None
        for cls in ["company-name", "itp-name", "employer-name"]:
            tag = card.find(class_=re.compile(cls, re.I))
            if tag:
                t = tag.find(string=True, recursive=False)
                if not t:
                    for child in tag.children:
                        if hasattr(child, 'get_text') and child.name != 'img':
                            t = child.get_text(strip=True); break
                if t and t.strip(): company = t.strip(); break
        if not company:
            img = card.find("img", alt=True)
            if img:
                alt = re.sub(r'\s*(Vietnam\s*)?(Small\s*)?Logo\s*$', '', img.get("alt",""), flags=re.I).strip()
                if alt: company = alt
        location = salary_text = None
        for cls in ["location", "city"]: 
            tag = card.find(class_=re.compile(cls, re.I))
            if tag: location = tag.get_text(strip=True); break
        for cls in ["salary", "iis-", "text-salary"]:
            tag = card.find(class_=re.compile(cls, re.I))
            if tag: salary_text = tag.get_text(strip=True); break
        skills = [t.get_text(strip=True) for t in card.find_all(class_=re.compile(r"tag|skill|badge", re.I))
                  if t.get_text(strip=True) and len(t.get_text(strip=True)) < 40
                  and t.get_text(strip=True) not in ("HOT","NEW","URGENT")]
        if title:
            jobs.append({
                "job_title": title, "company_name": company, "location": location,
                "salary": salary_text, "salary_min": None, "salary_max": None,
                "skills": skills or None, "description": None, "employment_type": None,
                "posting_date": datetime.now().strftime("%Y-%m-%d"),
                "source": f"ITviec (manual:{filename})", "url": url,
                "crawled_at": datetime.now().isoformat(),
            })
    return jobs

# ── JobsGO listing parser ─────────────────────────────────────
def parse_relative_date_vi(text: str) -> str:
    today = datetime.now().date()
    if not text: return today.strftime("%Y-%m-%d")
    text = text.strip().lower()
    m = re.search(r'(\d+)\s*(ph\u00fat|gi\u1edd)', text)
    if m: return today.strftime("%Y-%m-%d")
    m = re.search(r'(\d+)\s*ng\u00e0y', text)
    if m: return (today - timedelta(days=int(m.group(1)))).strftime("%Y-%m-%d")
    m = re.search(r'(\d+)\s*tu\u1ea7n', text)
    if m: return (today - timedelta(weeks=int(m.group(1)))).strftime("%Y-%m-%d")
    m = re.search(r'(\d+)\s*th\u00e1ng', text)
    if m: return (today - timedelta(days=int(m.group(1))*30)).strftime("%Y-%m-%d")
    return today.strftime("%Y-%m-%d")

def parse_jobsgo_listing(html: str, filename: str) -> list:
    soup = BeautifulSoup(html, "html.parser")
    cards = soup.find_all("div", class_=re.compile(r"\bjob-card\b"))
    jobs = []
    for card in cards:
        title_a = card.find("h3", class_="job-title")
        title_a = title_a.find("a") if title_a else card.find("a", title=True)
        title = title_a.get_text(strip=True) if title_a else None
        raw_url = title_a.get("href", "") if title_a else ""
        url = raw_url.split("?")[0] if raw_url else None
        comp_tag = card.find("a", class_="company-title")
        company = comp_tag.get_text(strip=True) if comp_tag else None
        if not company:
            img = card.find("img", alt=True)
            if img:
                alt = re.sub(r'\s*(Vietnam\s*)?(Small\s*)?Logo\s*$', '', img.get("alt",""), flags=re.I).strip()
                company = alt or None
        salary_text = location = None
        info_div = card.find("div", class_=re.compile(r"text-primary"))
        if info_div:
            spans = [s for s in info_div.find_all("span", recursive=False) if s.get_text(strip=True) not in ("|", "")]
            if spans: salary_text = spans[0].get_text(strip=True) or None
            if len(spans) >= 2: location = spans[-1].get_text(strip=True) or None
        employment_type = posted_rel = None
        for badge in card.find_all("span", class_="badge"):
            badge_title = badge.get("title", "")
            t = badge.get_text(strip=True)
            if "Lo\u1ea1i h\u00ecnh" in badge_title: employment_type = t
            elif "c\u1eadp nh\u1eadt" in badge_title.lower(): posted_rel = t
        posting_date = parse_relative_date_vi(posted_rel)
        salary_min = salary_max = None
        if salary_text:
            nums = [float(n.replace(",",".")) for n in re.findall(r'[\d]+(?:[.,]\d+)?', salary_text.replace(".",""))]
            if len(nums) >= 2: salary_min, salary_max = nums[0], nums[1]
            elif len(nums) == 1:
                if re.search(r"t\u1eeb|tr\u00ean", salary_text, re.I): salary_min = nums[0]
                else: salary_max = nums[0]
            for v in (salary_min, salary_max):
                if v and v > 1000:
                    salary_min = round(salary_min/1_000_000, 2) if salary_min else None
                    salary_max = round(salary_max/1_000_000, 2) if salary_max else None; break
        if title:
            jobs.append({
                "job_title": title, "company_name": company, "location": location,
                "salary": salary_text, "salary_min": salary_min, "salary_max": salary_max,
                "skills": None, "description": None, "employment_type": employment_type,
                "posting_date": posting_date, "source": f"JobsGO (manual:{filename})",
                "url": url, "crawled_at": datetime.now().isoformat(),
            })
    return jobs

# ── Vieclam24h listing parser ─────────────────────────────────
_VL24H_IT_PATHS = re.compile(r'/it-phan-mem/|/it-phan-cung-mang/', re.I)
_NON_IT_RE = re.compile(
    r"\b(telesale|kinh doanh|sale\b|k\u1ebf to\u00e1n|h\u00e0nh ch\u00ednh|"
    r"nh\u00e2n s\u1ef1|ch\u0103m s\u00f3c kh\u00e1ch h\u00e0ng|cskh|"
    r"t\u01b0 v\u1ea5n kh\u00e1ch h\u00e0ng|nh\u00e2n vi\u00ean v\u0103n ph\u00f2ng)", re.I)

def parse_vieclam24h_listing(html: str, filename: str) -> list:
    soup = BeautifulSoup(html, "html.parser")
    cards = soup.find_all("a", attrs={"data-job-id": True, "href": True})
    jobs = []
    for card in cards:
        raw_url = card.get("href", "")
        url = ("https://vieclam24h.vn" + raw_url.split("?")[0] if raw_url.startswith("/") else raw_url.split("?")[0])
        h3s = card.find_all("h3")
        title = h3s[0].get_text(strip=True) if h3s else None
        company = h3s[1].get_text(strip=True) if len(h3s) > 1 else None
        if not company:
            img = card.find("img", alt=True)
            if img: company = img.get("alt", "").strip() or None
        salary_text = location = None
        for span in card.find_all("span"):
            cls = " ".join(span.get("class", []))
            if "2C95FF" in cls or "text-primary" in cls.lower():
                t = span.get_text(strip=True)
                if t and any(c.isdigit() for c in t): salary_text = t; break
        for span in card.find_all("span"):
            cls = " ".join(span.get("class", []))
            if "neutral" in cls.lower() or "whitespace-nowrap" in cls:
                t = span.get_text(strip=True)
                if t and 2 < len(t) < 40 and not any(c.isdigit() for c in t): location = t; break
        posting_date = datetime.now().strftime("%Y-%m-%d")
        for div in card.find_all("div"):
            t = div.get_text(strip=True)
            m = re.search(r'h\u1ebft h\u1ea1n.*?(\d{2}/\d{2}/\d{4})', t)
            if m:
                try:
                    dl = datetime.strptime(m.group(1), "%d/%m/%Y")
                    posting_date = (dl - timedelta(days=30)).strftime("%Y-%m-%d")
                except Exception: pass
                break
        if not title: continue
        if _NON_IT_RE.search(title): continue
        if not _VL24H_IT_PATHS.search(raw_url): continue
        salary_min = salary_max = None
        if salary_text:
            nums = [float(n.replace(",",".")) for n in re.findall(r'\d+(?:[.,]\d+)?', salary_text.replace(".",""))]
            if len(nums) >= 2: salary_min, salary_max = nums[0], nums[1]
            elif len(nums) == 1:
                if re.search(r"t\u1eeb|tr\u00ean", salary_text, re.I): salary_min = nums[0]
                else: salary_max = nums[0]
            for v in (salary_min, salary_max):
                if v and v > 1000:
                    salary_min = round(salary_min/1_000_000, 2) if salary_min else None
                    salary_max = round(salary_max/1_000_000, 2) if salary_max else None; break
        jobs.append({
            "job_title": title, "company_name": company, "location": location,
            "salary": salary_text, "salary_min": salary_min, "salary_max": salary_max,
            "skills": None, "description": None, "employment_type": None,
            "posting_date": posting_date, "source": f"Vieclam24h (manual:{filename})",
            "url": url, "crawled_at": datetime.now().isoformat(),
        })
    return jobs

# ── Site detection + dispatch ─────────────────────────────────
def detect_site(html: str, filename: str) -> str:
    fn, head = filename.lower(), html[:10000].lower()
    if "itviec" in fn or "itviec.com" in head: return "itviec"
    if "jobsgo" in fn or "jobsgo.vn"  in head: return "jobsgo"
    if "vieclam24h" in fn or "vieclam24h.vn" in head: return "vieclam24h"
    if "topcv"  in fn or "topcv.vn"   in head: return "topcv"
    if "topdev" in fn or "topdev.vn"  in head: return "topdev"
    return "unknown"

def parse_file(path: Path) -> list:
    html     = path.read_text(encoding="utf-8", errors="replace")
    filename = path.name
    site     = detect_site(html, filename)
    print(f'  {filename} ({len(html):,} chars) → site: {site}')
    jld_items = extract_jsonld_jobs(html)
    if jld_items:
        jobs = [normalize_jsonld(j, site.capitalize()) for j in jld_items]
        print(f'    JSON-LD: {len(jobs)} JobPosting blocks'); return jobs
    if site == "itviec":     jobs = parse_itviec_listing(html, filename)
    elif site == "jobsgo":   jobs = parse_jobsgo_listing(html, filename)
    elif site == "vieclam24h": jobs = parse_vieclam24h_listing(html, filename)
    else: jobs = []
    print(f'    {site} listing: {len(jobs)} job cards'); return jobs

# ── CHẠY ─────────────────────────────────────────────────────
html_files = list(INPUT_DIR.glob("*.html")) + list(INPUT_DIR.glob("*.htm"))
if not html_files:
    print(f'Không tìm thấy file HTML trong {INPUT_DIR}/')
    print('Hướng dẫn: vào trang web → Ctrl+U → Ctrl+A → Ctrl+C → paste vào .html → lưu vào ../crawler/html_input/')
else:
    all_jobs = []
    for p in html_files:
        all_jobs.extend(parse_file(p))
    out_file = OUTPUT_DIR / "html_parsed.json"
    with open(out_file, "w", encoding="utf-8") as f:
        json.dump(all_jobs, f, ensure_ascii=False, indent=2)
    print(f'\n{"="*50}')
    print(f'Tổng: {len(all_jobs)} jobs extracted')
    print(f'  Có salary: {sum(1 for j in all_jobs if j.get("salary_min") or j.get("salary"))}')
    print(f'  Có skills: {sum(1 for j in all_jobs if j.get("skills"))}')
    print(f'\nSaved: {out_file}')

## **5. Gộp và loại trùng dữ liệu**

### **5.1. Mục tiêu**
Gộp tất cả 8 file JSON (từ 5 nguồn Wayback + VietJobs + HTML thủ công) thành **`all_jobs.json`** duy nhất, đồng thời loại bỏ job trùng lặp.

### **5.2. Chiến lược dedup**
* **Ưu tiên 1 — URL**: Nếu job có URL → dùng URL chuẩn hóa làm key. Cùng URL = trùng, giữ bản đầu tiên.
* **Ưu tiên 2 — title+company+tháng**: Cho VietJobs và HTML (không có URL). Ghép `title_normalized | company | YYYY-MM` làm key. Xử lý cross-site duplicates (cùng job đăng nhiều trang).

### **5.3. Thứ tự ưu tiên nguồn**
ITviec → TopDev → TopCV → Vieclam24h → VietJobs → HTML.
Nguồn đứng trước có độ tin cậy và chi tiết hơn → được giữ khi trùng.

### **5.4. Schema bộ dữ liệu (18 cột)**

| Cột | Kiểu | Mô tả | Nguồn |
|-----|------|-------|-------|
| `job_title` | str | Tên vị trí tuyển dụng | Tất cả |
| `company_name` | str | Tên công ty | Tất cả (trừ VietJobs) |
| `location` | str | Địa điểm (tỉnh/thành phố) | Tất cả |
| `salary` | str | Lương dạng text (VD: `15–25 triệu`) | Tất cả |
| `salary_min` | float | Lương tối thiểu (triệu VND) | Tất cả |
| `salary_max` | float | Lương tối đa (triệu VND) | Tất cả |
| `skills` | list→str | Kỹ năng yêu cầu (CSV: phân cách bằng dấu phẩy) | Tất cả |
| `description` | str | Mô tả công việc (tối đa 500 ký tự) | Wayback, VietJobs |
| `benefits` | str | Chế độ đãi ngộ | Wayback, VietJobs |
| `employment_type` | str | Loại hình (Full-time, Part-time, Contract...) | Tất cả |
| `industry` | str | Ngành nghề | VietJobs |
| `experience_required` | str | Mức kinh nghiệm yêu cầu | VietJobs, JobsGO |
| `posting_date` | str | Ngày đăng tin `YYYY-MM-DD` | Wayback, HTML |
| `snapshot_date` | str | Ngày Wayback lưu snapshot | Wayback |
| `deadline` | str | Hạn nộp hồ sơ `YYYY-MM-DD` | Wayback |
| `source` | str | Nguồn dữ liệu (ITviec, TopDev...) | Tất cả |
| `url` | str | Link gốc bài đăng | Tất cả (trừ VietJobs) |
| `crawled_at` | str | Thời điểm thu thập dữ liệu | Tất cả (trừ VietJobs) |

In [ ]:
"""
Section 5: Gộp tất cả file JSON thành all_jobs.json với dedup 2 cấp.
"""
import json
from pathlib import Path
from collections import Counter

OUTPUT_DIR = Path("../crawler/output")

SOURCE_FILES = [
    "wayback_itviec.json",
    "topdev.json",
    "wayback_topdev.json",
    "wayback_topcv.json",
    "wayback_jobsgo.json",
    "wayback_vieclam24h.json",
    "vietjobs_it.json",
    "html_parsed.json",
]

def load_json(path: Path) -> list:
    if not path.exists(): return []
    with open(path, encoding="utf-8") as f: data = json.load(f)
    return data if isinstance(data, list) else []

def dedup_key(job: dict) -> str:
    url = (job.get("url") or "").strip().rstrip("/").lower()
    if url: return f"url:{url}"
    title   = (job.get("job_title") or "").strip().lower()
    company = (job.get("company_name") or "").strip().lower()
    date    = (job.get("posting_date") or "")[:7]
    return f"title:{title}|{company}|{date}"

# ── CHẠY ─────────────────────────────────────────────────────
seen: dict = {}
source_counts: dict = {}

for fname in SOURCE_FILES:
    jobs = load_json(OUTPUT_DIR / fname)
    added = 0
    for job in jobs:
        key = dedup_key(job)
        if key not in seen:
            seen[key] = job; added += 1
    source_counts[fname] = added
    if jobs:
        print(f'  {fname}: {len(jobs)} total, {added} new unique')

# Gộp thêm html_parsed_*.json nếu có
for path in sorted(OUTPUT_DIR.glob("html_parsed*.json")):
    if path.name in SOURCE_FILES: continue
    jobs = load_json(path); added = 0
    for job in jobs:
        key = dedup_key(job)
        if key not in seen: seen[key] = job; added += 1
    if jobs: print(f'  {path.name}: {len(jobs)} total, {added} new unique')

all_jobs = list(seen.values())

out_path = OUTPUT_DIR / "all_jobs.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(all_jobs, f, ensure_ascii=False, indent=2)

has_salary = sum(1 for j in all_jobs if j.get("salary_min") or j.get("salary"))
has_skills = sum(1 for j in all_jobs if j.get("skills"))
has_date   = sum(1 for j in all_jobs if j.get("posting_date"))

months = Counter(
    j["posting_date"][:7]
    for j in all_jobs
    if j.get("posting_date") and len(j["posting_date"]) >= 7
)
bar_max = max(months.values()) if months else 1

print(f'\n{"="*55}')
print(f'MERGED DATASET: {len(all_jobs)} unique IT jobs')
print(f'  Has salary : {has_salary} ({has_salary*100//len(all_jobs)}%)')
print(f'  Has skills : {has_skills} ({has_skills*100//len(all_jobs)}%)')
print(f'  Has date   : {has_date}')
print(f'\nMonthly distribution ({len(months)} months):')
for ym, cnt in sorted(months.items()):
    bar = '\u2588' * min(40, cnt * 40 // bar_max)
    print(f'  {ym}: {cnt:4d} {bar}')
print(f'\nSaved: {out_path}')

# ── Xuất CSV vào data/ ───────────────────────────────────────
import pandas as pd

DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

df = pd.DataFrame(all_jobs)

# skills là list → chuyển sang chuỗi phân cách bằng dấu phẩy
df["skills"] = df["skills"].apply(
    lambda x: ", ".join(x) if isinstance(x, list) else x
)

csv_path = DATA_DIR / "vietnam_it_jobs_raw.csv"
df.to_csv(csv_path, index=False, encoding="utf-8-sig")

print(f'\nCSV exported: {csv_path}')
print(f'  Rows: {len(df):,} | Columns: {len(df.columns)}')
print(f'  Columns: {list(df.columns)}')

## **6. Đánh giá chất lượng bộ dữ liệu**

### **6.1. Mục tiêu**
Kiểm tra bộ dữ liệu cuối cùng trên 4 chiều để đảm bảo đủ dữ liệu cho các mục tiêu phân tích:

| Mục tiêu phân tích | Trường cần | Ngưỡng đủ |
|---------------------|-----------|----------|
| Xu hướng theo tháng | `posting_date` | ≥ 5,000 jobs có date |
| Phân tích lương | `salary_min`/`salary_max` | ≥ 2,000 jobs có số |
| Top skills | `skills` | ≥ 5,000 jobs có skills |
| Bản đồ địa lý | `location` | ≥ 5,000 jobs có location |

### **6.2. Thống kê đầu ra**
Cell dưới sẽ in báo cáo đầy đủ và vẽ biểu đồ phân bố dữ liệu theo tháng và theo nguồn.

In [ ]:
"""
Section 6: Đánh giá chất lượng bộ dữ liệu all_jobs.json.
"""
import json
from pathlib import Path
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

out_path = Path("../crawler/output/all_jobs.json")
with open(out_path, encoding="utf-8") as f:
    all_jobs = json.load(f)

df = pd.DataFrame(all_jobs)
print(f'Tổng số jobs: {len(df):,}')
print(f'Số cột       : {len(df.columns)}')
print(f'Cột          : {list(df.columns)}\n')

# ── Tỉ lệ điền đầy đủ ────────────────────────────────────────
print('=== TỶ LỆ ĐẦY ĐỦ DỮ LIỆU ===')
checks = [
    ('posting_date',  df['posting_date'].notna().sum(),             'Xu hướng theo tháng'),
    ('salary_min',    df['salary_min'].notna().sum(),               'Phân tích lương (số)'),
    ('salary',        df['salary'].notna().sum(),                   'Có text lương'),
    ('skills',        df['skills'].apply(lambda x: bool(x)).sum(), 'Top skills'),
    ('location',      df['location'].notna().sum(),                 'Bản đồ địa lý'),
]
for col, cnt, desc in checks:
    pct = cnt * 100 // len(df)
    bar = '█' * (pct // 5)
    print(f'  {desc:<30} {cnt:>5} ({pct:>2}%) {bar}')

# ── Phân bố theo nguồn ───────────────────────────────────────
print('\n=== PHÂN BỐ THEO NGUỒN ===')
src_counts = Counter(j.get('source','').split(' ')[0] for j in all_jobs)
for src, cnt in src_counts.most_common():
    print(f'  {src:<15} {cnt:>5}')

# ── Phân bố theo tháng ───────────────────────────────────────
months = Counter(
    j['posting_date'][:7]
    for j in all_jobs
    if j.get('posting_date') and len(j.get('posting_date','')) >= 7
)
print(f'\n=== PHÂN BỐ THEO THÁNG ({len(months)} tháng) ===')
bar_max = max(months.values()) if months else 1
for ym, cnt in sorted(months.items()):
    bar = '█' * min(40, cnt * 40 // bar_max)
    print(f'  {ym}: {cnt:4d} {bar}')

# ── Biểu đồ ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Dataset Overview — Vietnam IT Jobs', fontsize=14, fontweight='bold')

# Chart 1: Phân bố theo nguồn
src_names = [s for s, _ in src_counts.most_common()]
src_vals  = [c for _, c in src_counts.most_common()]
axes[0].barh(src_names, src_vals, color='steelblue')
axes[0].set_title('Jobs by Source')
axes[0].set_xlabel('Number of jobs')
for i, v in enumerate(src_vals):
    axes[0].text(v + 20, i, str(v), va='center')

# Chart 2: Phân bố theo tháng
if months:
    ym_sorted = sorted(months.keys())
    counts    = [months[ym] for ym in ym_sorted]
    axes[1].bar(range(len(ym_sorted)), counts, color='darkorange')
    axes[1].set_xticks(range(0, len(ym_sorted), max(1, len(ym_sorted)//10)))
    axes[1].set_xticklabels([ym_sorted[i] for i in range(0, len(ym_sorted), max(1, len(ym_sorted)//10))],
                             rotation=45, ha='right', fontsize=8)
    axes[1].set_title('Jobs by Month')
    axes[1].set_xlabel('Month')
    axes[1].set_ylabel('Number of jobs')

plt.tight_layout()
plt.savefig('../crawler/output/dataset_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print('\nChart saved: ../crawler/output/dataset_overview.png')